# Hardware-Native Hardened Posiform QUBO 생성

## 논문 재현: Pelofske, Hahn, Djidjev (2024)

논문의 hardware-native QUBO 생성 파이프라인을 재현한다.

**기존 구현과의 차이:**

| | 기존 (posiform.ipynb) | 본 노트북 (논문 방식) |
|---|---|---|
| 그래프 | complete graph (모든 쌍 연결) | **hardware topology** (Chimera/Pegasus/Zephyr) |
| 분할 | 순차 균등 분할 | **Kernighan-Lin recursive bisection** |
| Random QUBO | complete subgraph | **hardware subgraph 에지만** |
| Posiform | complete graph 위 clause | **hardware 에지 위 clause만** |
| QPU 실행 | embedding 필요 | **embedding 불필요** (이미 hardware-native) |

**파이프라인:**
1. Hardware graph 생성 (Chimera C16 / Pegasus P16 / Zephyr Z4,Z6)
2. KL recursive bisection → disjoint subgraph 분할
3. 각 subgraph에 discrete-coefficient random QUBO 생성 (hardware edge만)
4. 각 subproblem GS brute force
5. Concatenate → planted target
6. Posiform planting (hardware edge만)
7. `Q_final = Σ R_i + α × P`

In [1]:
import sys
!{sys.executable} -m pip install dwave-system dwave-neal dwave-networkx numpy matplotlib python-sat

In [3]:
import numpy as np
import networkx as nx
import dwave_networkx as dnx
from networkx.algorithms.community import kernighan_lin_bisection
from itertools import product
from pysat.solvers import Minisat22
import random
import time

np.set_printoptions(threshold=200, linewidth=150, precision=3, suppress=True)

COEFF_LIN2 = [-1, 1]
COEFF_LIN20 = [round(-1 + 0.1 * i, 1) for i in range(21)]

print("imports OK")

imports OK


## 1. Hardware Topology 생성

| Topology | 그래프 | Nodes | Edges | Max Degree |
|---|---|---|---|---|
| Chimera C16 | `chimera_graph(16)` | 2048 | 6016 | 6 |
| Pegasus P16 | `pegasus_graph(16)` | 5640 | 40484 | 15 |
| Zephyr Z4 | `zephyr_graph(4)` | 576 | 5032 | 20 |
| Zephyr Z6 | `zephyr_graph(6)` | 1248 | 11400 | 20 |

In [4]:
def get_hardware_graph(topology='pegasus', size=None):
    """
    D-Wave hardware topology 그래프 생성.

    Args:
        topology: 'chimera', 'pegasus', 'zephyr'
        size: topology 파라미터 (None이면 기본값)

    Returns:
        G: networkx.Graph (노드 = 큐빗, 에지 = 커플러)
    """
    if topology == 'chimera':
        m = size or 16
        G = dnx.chimera_graph(m)
        name = f'Chimera C{m}'
    elif topology == 'pegasus':
        m = size or 16
        G = dnx.pegasus_graph(m)
        name = f'Pegasus P{m}'
    elif topology == 'zephyr':
        m = size or 4
        G = dnx.zephyr_graph(m)
        name = f'Zephyr Z{m}'
    else:
        raise ValueError(f"Unknown topology: {topology}")

    degrees = [d for _, d in G.degree()]
    print(f"[{name}] nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, "
          f"degree: min={min(degrees)}, max={max(degrees)}, avg={np.mean(degrees):.1f}")
    return G, name


# 테스트
for topo, sz in [('chimera', 16), ('pegasus', 16), ('zephyr', 4), ('zephyr', 6)]:
    get_hardware_graph(topo, sz)
    print()

[Chimera C16] nodes=2048, edges=6016, degree: min=5, max=6, avg=5.9

[Pegasus P16] nodes=5640, edges=40484, degree: min=6, max=15, avg=14.4

[Zephyr Z4] nodes=576, edges=5032, degree: min=10, max=20, avg=17.5

[Zephyr Z6] nodes=1248, edges=11400, degree: min=10, max=20, avg=18.3



## 2. Kernighan-Lin Recursive Bisection

논문 Section 2.2: "we partition the hardware graph into disjoint induced subgraphs using Kernighan-Lin recursive bisection"

재귀적으로 이등분하여 각 partition이 `max_sub_size` 이하가 될 때까지 분할.

In [5]:
def kl_recursive_bisection(G, max_sub_size=15):
    """
    Kernighan-Lin recursive bisection으로 그래프를 disjoint subgraph로 분할.

    논문 방식: 각 partition이 max_sub_size 이하가 될 때까지 재귀적으로 이등분.

    Args:
        G: networkx.Graph
        max_sub_size: 각 partition 최대 크기

    Returns:
        partitions: [set of nodes, ...] — disjoint partition 리스트
    """
    nodes = set(G.nodes())
    if len(nodes) <= max_sub_size:
        return [nodes]

    # KL bisection
    sub = G.subgraph(nodes).copy()
    try:
        A, B = kernighan_lin_bisection(sub)
    except nx.NetworkXError:
        # 연결되지 않은 그래프면 connected component별로 재귀
        partitions = []
        for comp in nx.connected_components(sub):
            partitions.extend(kl_recursive_bisection(G.subgraph(comp).copy(), max_sub_size))
        return partitions

    # 재귀
    result = []
    for part in [A, B]:
        if len(part) <= max_sub_size:
            result.append(part)
        else:
            result.extend(kl_recursive_bisection(G.subgraph(part).copy(), max_sub_size))

    return result


# 테스트: Pegasus P16
G_test, _ = get_hardware_graph('pegasus', 16)
t0 = time.time()
parts = kl_recursive_bisection(G_test, max_sub_size=15)
elapsed = time.time() - t0

sizes = [len(p) for p in parts]
print(f"\n[KL Bisection 결과] {elapsed:.2f}s")
print(f"  partitions: {len(parts)}")
print(f"  sizes: min={min(sizes)}, max={max(sizes)}, avg={np.mean(sizes):.1f}")
print(f"  total nodes: {sum(sizes)} / {G_test.number_of_nodes()}")
print(f"  size 분포: {dict(sorted(zip(*np.unique(sizes, return_counts=True))))}")

[Pegasus P16] nodes=5640, edges=40484, degree: min=6, max=15, avg=14.4

[KL Bisection 결과] 2.24s
  partitions: 512
  sizes: min=11, max=12, avg=11.0
  total nodes: 5640 / 5640
  size 분포: {np.int64(11): np.int64(504), np.int64(12): np.int64(8)}


## 3. Hardware-Native Random QUBO + Posiform 생성

핵심: **hardware graph의 에지에만** 계수를 배정.
complete graph와 달리, 존재하지 않는 에지에는 coupling이 없다.

In [6]:
def gen_hardware_random_qubo(G, variables, coeff_type='lin2'):
    """
    Hardware subgraph 위에 random discrete-coefficient QUBO 생성.
    논문 Step 2: hardware edge에만 coupling 배정.
    """
    coeffs = COEFF_LIN2 if coeff_type == 'lin2' else COEFF_LIN20
    var_set = set(variables)
    Q = {}

    for v in variables:
        Q[(v, v)] = random.choice(coeffs)

    sub = G.subgraph(var_set)
    for i, j in sub.edges():
        lo, hi = min(i, j), max(i, j)
        Q[(lo, hi)] = random.choice(coeffs)

    return Q


def solve_subproblem_brute_force(Q, variables):
    """
    Brute force로 subproblem QUBO의 GS 계산.
    GS의 모든 assignment + first excited state 에너지도 반환.

    Returns:
        best_assignment: {node: 0 or 1} (첫 번째 GS)
        best_energy: float
        num_degenerate: int
        all_gs_assignments: [{node: 0 or 1}, ...] (모든 GS)
        first_excited_energy: float (GS 다음으로 낮은 에너지, 없으면 inf)
    """
    n = len(variables)
    if n > 23:
        raise ValueError(f"Brute force 불가: subgraph size {n} > 23")

    var_list = sorted(variables)
    var_to_idx = {v: i for i, v in enumerate(var_list)}

    linear = []
    quad = []
    for (i, j), w in Q.items():
        if i == j:
            linear.append((var_to_idx[i], w))
        else:
            quad.append((var_to_idx[i], var_to_idx[j], w))

    # linear 중복 제거
    linear_map = {}
    for idx, w in linear:
        linear_map[idx] = linear_map.get(idx, 0) + w
    linear = [(idx, w) for idx, w in linear_map.items() if w != 0]
    quad = [(a, b, w) for a, b, w in quad if w != 0]

    best_energy = float('inf')
    first_excited_energy = float('inf')
    best_bits = 0
    num_degenerate = 0
    all_gs_bits = []

    for bits in range(1 << n):
        energy = 0.0
        for idx, w in linear:
            if (bits >> (n - 1 - idx)) & 1:
                energy += w
        for a, b, w in quad:
            if ((bits >> (n - 1 - a)) & 1) and ((bits >> (n - 1 - b)) & 1):
                energy += w

        if energy < best_energy - 1e-12:
            # 이전 best가 first excited가 됨
            if num_degenerate > 0:
                first_excited_energy = best_energy
            best_energy = energy
            best_bits = bits
            num_degenerate = 1
            all_gs_bits = [bits]
        elif abs(energy - best_energy) < 1e-12:
            num_degenerate += 1
            all_gs_bits.append(bits)
        elif energy < first_excited_energy - 1e-12:
            first_excited_energy = energy

    # assignment 변환
    def bits_to_assignment(bits):
        return {v: (bits >> (n - 1 - i)) & 1 for i, v in enumerate(var_list)}

    best_assignment = bits_to_assignment(best_bits)
    all_gs_assignments = [bits_to_assignment(b) for b in all_gs_bits]

    return (best_assignment, best_energy, num_degenerate,
            all_gs_assignments, first_excited_energy)


def compute_delta_p(Q_posiform, target_assignment, all_block_gs, partitions):
    """
    Δ_P 계산: R-축퇴 상태들 중 posiform 에너지가 최소인 것.

    각 축퇴 블록의 대안 GS를 하나씩 교체하여 P(y)를 계산.
    단일 블록 교체만 고려 (가장 작은 Δ_P를 줄 가능성이 높음).

    Args:
        Q_posiform: {(i,j): weight} — posiform QUBO (α=1 스케일)
        target_assignment: {node: 0 or 1}
        all_block_gs: [block별 [assignment1, assignment2, ...]]
        partitions: [set of nodes, ...]

    Returns:
        delta_p: float (최소 P(y), y ∈ GS(R)\{x*})
        num_degenerate_blocks: 축퇴 블록 수
    """
    def eval_posiform(assignment):
        """assignment에서 posiform 에너지 계산."""
        energy = 0.0
        for (i, j), w in Q_posiform.items():
            xi = assignment.get(i, 0)
            if i == j:
                energy += w * xi
            else:
                xj = assignment.get(j, 0)
                energy += w * xi * xj
        return energy

    min_p = float('inf')
    num_degenerate_blocks = 0

    for block_idx, (gs_list, part) in enumerate(zip(all_block_gs, partitions)):
        if len(gs_list) <= 1:
            continue  # 축퇴 없는 블록은 건너뜀
        num_degenerate_blocks += 1

        # target의 이 블록 assignment
        target_block = {v: target_assignment[v] for v in part}

        # 대안 GS들에 대해 P(y) 계산
        for alt_gs in gs_list:
            # target과 같은 assignment면 건너뜀
            if all(alt_gs[v] == target_block[v] for v in part):
                continue

            # y = target에서 이 블록만 alt_gs로 교체
            y = dict(target_assignment)
            y.update(alt_gs)

            # P(y) - P(x*) = delta (상수항 상쇄)
            p_y = eval_posiform(y) - eval_posiform(target_assignment)
            min_p = min(min_p, p_y)

    return min_p, num_degenerate_blocks


def compute_delta_r(block_energies):
    """
    Δ_R 계산: R의 최소 비축퇴 에너지 갭.
    = min over all blocks of (first_excited - ground_state).

    Args:
        block_energies: [(gs_energy, first_excited_energy), ...]

    Returns:
        delta_r: float
    """
    delta_r = float('inf')
    for gs_e, fe_e in block_energies:
        if fe_e < float('inf'):
            gap = fe_e - gs_e
            if gap > 1e-12:
                delta_r = min(delta_r, gap)
    return delta_r


print("확장된 brute force + Δ_P, Δ_R 계산 함수 정의 완료")

확장된 brute force + Δ_P, Δ_R 계산 함수 정의 완료


In [7]:
def posiform_planting_hardware(G, target_assignment, alpha):
    """
    Hardware graph 위에 posiform planting.
    논문 Step 5: hardware edge에만 clause 생성.

    MiniSat으로 2-SAT uniqueness 검증.

    Args:
        G: hardware graph
        target_assignment: {node: 0 or 1}
        alpha: posiform 계수 (스케일링)

    Returns:
        Q_posiform: {(i,j): weight} — posiform QUBO
    """
    nodes = sorted(target_assignment.keys())
    n = len(nodes)
    node_to_idx = {v: i for i, v in enumerate(nodes)}
    edges = [(min(u, v), max(u, v)) for u, v in G.subgraph(nodes).edges()]

    if not edges:
        return {}

    Q_posiform = {}
    clauses_cnf = []
    all_tuples = [(0, 0), (0, 1), (1, 0), (1, 1)]
    max_clauses = 1000 * n

    def add_posiform_term(i, j):
        target_tuple = (target_assignment[i], target_assignment[j])
        wrong_tuples = [t for t in all_tuples if t != target_tuple]
        wi, wj = random.choice(wrong_tuples)
        lo, hi = min(i, j), max(i, j)

        if wi == 0 and wj == 0:
            Q_posiform[(i, i)] = Q_posiform.get((i, i), 0) - alpha
            Q_posiform[(j, j)] = Q_posiform.get((j, j), 0) - alpha
            Q_posiform[(lo, hi)] = Q_posiform.get((lo, hi), 0) + alpha
        elif wi == 0 and wj == 1:
            Q_posiform[(j, j)] = Q_posiform.get((j, j), 0) + alpha
            Q_posiform[(lo, hi)] = Q_posiform.get((lo, hi), 0) - alpha
        elif wi == 1 and wj == 0:
            Q_posiform[(i, i)] = Q_posiform.get((i, i), 0) + alpha
            Q_posiform[(lo, hi)] = Q_posiform.get((lo, hi), 0) - alpha
        else:
            Q_posiform[(lo, hi)] = Q_posiform.get((lo, hi), 0) + alpha

        # CNF clause
        lit_i = (node_to_idx[i] + 1) if wi == 0 else -(node_to_idx[i] + 1)
        lit_j = (node_to_idx[j] + 1) if wj == 0 else -(node_to_idx[j] + 1)
        clauses_cnf.append([lit_i, lit_j])

    def check_uniqueness():
        with Minisat22() as solver:
            for clause in clauses_cnf:
                solver.add_clause(clause)
            blocking = []
            for node in nodes:
                idx = node_to_idx[node] + 1
                blocking.append(-idx if target_assignment[node] == 1 else idx)
            solver.add_clause(blocking)
            return not solver.solve()

    check_interval = max(1, n // 4)

    for step in range(max_clauses):
        # hardware edge에서 랜덤 선택
        i, j = random.choice(edges)
        add_posiform_term(i, j)

        if (step + 1) % check_interval == 0:
            if check_uniqueness():
                print(f"  [posiform] {step + 1} clauses로 유일성 확보")
                Q_posiform = {k: v for k, v in Q_posiform.items() if abs(v) > 1e-15}
                return Q_posiform

    print(f"  [Warning] {max_clauses} clauses 후에도 유일성 미확보")
    Q_posiform = {k: v for k, v in Q_posiform.items() if abs(v) > 1e-15}
    return Q_posiform


print("Posiform planting 함수 정의 완료")

Posiform planting 함수 정의 완료


## 4. 전체 파이프라인: Hardware-Native Hardened Posiform

In [8]:
def gen_hardware_native_qubo(topology='pegasus', topo_size=None,
                              max_sub_size=15, coeff_type='lin2',
                              posiform_scale=0.1, seed=None):
    """
    Hardware-Native Hardened Posiform QUBO 생성 (논문 전체 파이프라인).
    """
    if seed is not None:
        random.seed(seed)

    G, topo_name = get_hardware_graph(topology, topo_size)

    t0 = time.time()
    partitions = kl_recursive_bisection(G, max_sub_size)
    bisection_time = time.time() - t0
    print(f"  [KL bisection] {len(partitions)} partitions, {bisection_time:.2f}s")

    t0 = time.time()
    random_qubos = []
    target_assignment = {}
    total_random_energy = 0.0
    total_degenerate = 1

    for part in partitions:
        variables = sorted(part)
        R = gen_hardware_random_qubo(G, variables, coeff_type)
        assignment, energy, deg, _, _ = solve_subproblem_brute_force(R, variables)

        target_assignment.update(assignment)
        random_qubos.append(R)
        total_random_energy += energy
        total_degenerate *= deg

    bf_time = time.time() - t0
    print(f"  [Random QUBO + BF] {bf_time:.2f}s, "
          f"total deg={total_degenerate}")

    target_str = ''.join(str(target_assignment[n]) for n in sorted(target_assignment))

    t0 = time.time()
    Q_posiform = posiform_planting_hardware(G, target_assignment, posiform_scale)
    posiform_time = time.time() - t0
    print(f"  [Posiform] {posiform_time:.2f}s, {len(Q_posiform)} terms")

    Q_final = {}
    for R in random_qubos:
        for key, val in R.items():
            Q_final[key] = Q_final.get(key, 0) + val
    for key, val in Q_posiform.items():
        Q_final[key] = Q_final.get(key, 0) + val
    Q_final = {k: v for k, v in Q_final.items() if abs(v) > 1e-15}

    target_energy = 0.0
    for (i, j), w in Q_final.items():
        xi = target_assignment.get(i, 0)
        xj = target_assignment.get(j, 0) if i != j else 1
        if i == j:
            target_energy += w * xi
        else:
            target_energy += w * xi * xj

    info = {
        'topology': topo_name,
        'n': len(target_assignment),
        'num_partitions': len(partitions),
        'partition_sizes': sorted([len(p) for p in partitions]),
        'coeff_type': coeff_type,
        'posiform_scale': posiform_scale,
        'total_degeneracy': total_degenerate,
        'random_total_energy': total_random_energy,
        'target_energy': target_energy,
        'num_qubo_terms': len(Q_final),
        'num_edges_hardware': G.number_of_edges(),
    }

    return Q_final, target_assignment, info


print("전체 파이프라인 함수 정의 완료")

전체 파이프라인 함수 정의 완료


## 5. QUBO 생성 테스트 (각 토폴로지)

In [9]:
# ═══ 각 토폴로지에서 QUBO 생성 테스트 ═══
configs = [
    ('chimera', 16, 'lin2', 0.1),
    ('pegasus', 16, 'lin2', 0.1),
    ('zephyr', 4, 'lin2', 0.1),
]

results = {}
for topo, sz, coeff, alpha in configs:
    print(f"\n{'═' * 60}")
    print(f"  {topo.upper()} (size={sz}, coeff={coeff}, α={alpha})")
    print(f"{'═' * 60}")

    t0 = time.time()
    Q, target_assign, info = gen_hardware_native_qubo(
        topology=topo, topo_size=sz,
        max_sub_size=15, coeff_type=coeff,
        posiform_scale=alpha, seed=42
    )
    total_time = time.time() - t0

    print(f"\n  [결과]")
    print(f"    변수 수: {info['n']}")
    print(f"    partitions: {info['num_partitions']}")
    print(f"    partition 크기: min={min(info['partition_sizes'])}, "
          f"max={max(info['partition_sizes'])}")
    print(f"    QUBO 항 수: {info['num_qubo_terms']}")
    print(f"    Target energy: {info['target_energy']:.2f}")
    print(f"    총 축퇴도: {info['total_degeneracy']}")
    print(f"    총 생성 시간: {total_time:.2f}s")

    results[topo] = {'Q': Q, 'target': target_assign, 'info': info}


════════════════════════════════════════════════════════════
  CHIMERA (size=16, coeff=lin2, α=0.1)
════════════════════════════════════════════════════════════
[Chimera C16] nodes=2048, edges=6016, degree: min=5, max=6, avg=5.9
  [KL bisection] 256 partitions, 0.35s
  [Random QUBO + BF] 0.08s, total deg=8062789073834749552826668737703796969479925768242378563806822400000000000000000000000
  [posiform] 20480 clauses로 유일성 확보
  [Posiform] 0.12s, 6790 terms

  [결과]
    변수 수: 2048
    partitions: 256
    partition 크기: min=8, max=8
    QUBO 항 수: 7635
    Target energy: -1987.10
    총 축퇴도: 8062789073834749552826668737703796969479925768242378563806822400000000000000000000000
    총 생성 시간: 0.55s

════════════════════════════════════════════════════════════
  PEGASUS (size=16, coeff=lin2, α=0.1)
════════════════════════════════════════════════════════════
[Pegasus P16] nodes=5640, edges=40484, degree: min=6, max=15, avg=14.4
  [KL bisection] 512 partitions, 2.20s
  [Random QUBO + BF] 1.90s, tota

## 6. SA 검증

생성된 hardware-native QUBO를 SA로 풀어서 GS 검증.

In [10]:
import neal

sa_sampler = neal.SimulatedAnnealingSampler()

print(f"{'Topology':<12} {'N':>6} {'α':>5} {'SA Best E':>12} {'Target E':>12} {'Match':>6}")
print(f"{'─' * 55}")

for topo in ['chimera', 'pegasus', 'zephyr']:
    r = results[topo]
    Q = r['Q']
    target = r['target']
    info = r['info']
    n = info['n']
    target_energy = info['target_energy']

    resp = sa_sampler.sample_qubo(Q, num_reads=100, num_sweeps=5000)

    sorted_nodes = sorted(target.keys())
    eng_ok = 0
    best_e = float('inf')

    for sample, energy, _ in resp.data(['sample', 'energy', 'num_occurrences']):
        best_e = min(best_e, energy)
        if abs(energy - target_energy) < 1e-4:
            eng_ok += 1

    print(f"{info['topology']:<12} {n:>6} {info['posiform_scale']:>5} "
          f"{best_e:>12.2f} {target_energy:>12.2f} {eng_ok:>5}%")

Topology          N     α    SA Best E     Target E  Match
───────────────────────────────────────────────────────
Chimera C16    2048   0.1     -1987.10     -1987.10   100%
Pegasus P16    5640   0.1     -7817.70     -7817.70   100%
Zephyr Z4       576   0.1      -626.80      -626.80   100%


## 7. 인스턴스 사전 생성 + 저장/로드

인스턴스 생성(KL bisection + brute force + posiform)이 오래 걸리므로, 한 번 만들어 pickle로 저장해두고 실험마다 재활용.

- `instances_pegasus16_lin2_500.pkl` — 500개
- `instances_pegasus16_lin2_1000.pkl` — 1000개

In [15]:
# ═══ 인스턴스 사전 생성 + 저장 (병렬화) ═══
import pickle
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# ─── 설정 ───
INST_TOPO = 'zephyr'
INST_TOPO_SIZE = 4
INST_COEFF = 'lin20'
INST_MAX_SUB = 15
INST_COUNTS = [500]  # 생성할 인스턴스 수 목록
NUM_WORKERS = os.cpu_count()  # CPU 코어 수만큼 병렬화

inst_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                        'hardened_posiform', 'instances')
os.makedirs(inst_dir, exist_ok=True)


def _generate_single_instance(inst, G, max_sub, coeff):
    """단일 인스턴스 생성 (워커 함수)."""
    random.seed(inst * 53)
    parts = kl_recursive_bisection(G, max_sub)

    random_qubos = []
    target = {}
    total_deg = 1
    all_block_gs = []
    block_energies = []

    for part in parts:
        variables = sorted(part)
        R = gen_hardware_random_qubo(G, variables, coeff)
        assignment, energy, deg, gs_assignments, fe_energy = solve_subproblem_brute_force(R, variables)
        target.update(assignment)
        random_qubos.append(R)
        total_deg *= deg
        all_block_gs.append(gs_assignments)
        block_energies.append((energy, fe_energy))

    P = posiform_planting_hardware(G, target, 1.0)

    delta_p, num_deg_blocks = compute_delta_p(P, target, all_block_gs, parts)
    delta_r = compute_delta_r(block_energies)

    # R_sum 사전 계산
    R_sum = {}
    for R in random_qubos:
        for k, v in R.items():
            R_sum[k] = R_sum.get(k, 0) + v

    t_energy_r = sum(w * target.get(i, 0) * (target.get(j, 0) if i != j else 1)
                     if i != j else w * target.get(i, 0)
                     for (i, j), w in R_sum.items())
    t_energy_p = sum(w * target.get(i, 0) * (target.get(j, 0) if i != j else 1)
                     if i != j else w * target.get(i, 0)
                     for (i, j), w in P.items())

    sorted_nodes = sorted(target.keys())
    target_str = ''.join(str(target[nd]) for nd in sorted_nodes)

    return {
        'R_sum': R_sum,
        'P': P,
        'target': target,
        'target_str': target_str,
        'sorted_nodes': sorted_nodes,
        'n': len(target),
        't_energy_r': t_energy_r,
        't_energy_p': t_energy_p,
        'delta_p': delta_p,
        'delta_r': delta_r,
        'total_degeneracy': total_deg,
        'num_degenerate_blocks': num_deg_blocks,
        'seed': inst * 53,
    }


def generate_and_save_instances(num_instances, topo, topo_size, coeff, max_sub):
    """인스턴스 병렬 생성 + Δ_P, Δ_R 계산 + pickle 저장."""
    fname = f'instances_{topo}{topo_size}_{coeff}_{num_instances}.pkl'
    fpath = os.path.join(inst_dir, fname)

    # 이미 존재하면 건너뜀
    if os.path.exists(fpath):
        print(f"  [{fname}] 이미 존재 — 건너뜀")
        return fpath

    print(f"  [{fname}] 생성 중... ({num_instances}개, workers={NUM_WORKERS})")
    G, topo_name = get_hardware_graph(topo, topo_size)

    t0 = time.time()
    worker_fn = partial(_generate_single_instance, G=G, max_sub=max_sub, coeff=coeff)

    all_instances = [None] * num_instances
    done_count = 0

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(worker_fn, inst): inst
                   for inst in range(num_instances)}

        for future in as_completed(futures):
            inst_idx = futures[future]
            all_instances[inst_idx] = future.result()
            done_count += 1
            if done_count % 100 == 0:
                elapsed = time.time() - t0
                print(f"    {done_count}/{num_instances} ({elapsed:.0f}s)")

    # 메타정보 포함하여 저장
    save_obj = {
        'meta': {
            'topology': topo, 'topo_size': topo_size,
            'coeff': coeff, 'max_sub_size': max_sub,
            'num_instances': num_instances,
            'n': all_instances[0]['n'],
            'delta_p_mean': float(np.mean([d['delta_p'] for d in all_instances])),
            'delta_r_mean': float(np.mean([d['delta_r'] for d in all_instances])),
        },
        'instances': all_instances,
    }

    with open(fpath, 'wb') as f:
        pickle.dump(save_obj, f)

    elapsed = time.time() - t0
    fsize = os.path.getsize(fpath) / 1024 / 1024
    print(f"    저장 완료: {fpath} ({fsize:.1f} MB, {elapsed:.0f}s)")
    print(f"    Δ_P mean={save_obj['meta']['delta_p_mean']:.4f}, "
          f"Δ_R mean={save_obj['meta']['delta_r_mean']:.2f}")
    return fpath


def load_instances(num_instances, topo='pegasus', topo_size=16, coeff='lin2'):
    """저장된 인스턴스 로드."""
    fname = f'instances_{topo}{topo_size}_{coeff}_{num_instances}.pkl'
    fpath = os.path.join(inst_dir, fname)
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"{fpath} 없음 — 먼저 생성하세요")
    with open(fpath, 'rb') as f:
        data = pickle.load(f)
    meta = data['meta']
    print(f"  로드: {fname} ({meta['num_instances']}개, n={meta['n']}, "
          f"Δ_P={meta['delta_p_mean']:.4f}, Δ_R={meta['delta_r_mean']:.2f})")
    return data['instances'], data['meta']


# ─── 생성 실행 ───
print(f"  CPU 코어 수: {NUM_WORKERS}")
for count in INST_COUNTS:
    generate_and_save_instances(count, INST_TOPO, INST_TOPO_SIZE, INST_COEFF, INST_MAX_SUB)

print(f"\n  인스턴스 디렉토리: {inst_dir}")
print(f"  파일 목록:")
for f in sorted(os.listdir(inst_dir)):
    fsize = os.path.getsize(os.path.join(inst_dir, f)) / 1024 / 1024
    print(f"    {f} ({fsize:.1f} MB)")

  CPU 코어 수: 28
  [instances_zephyr4_lin20_500.pkl] 생성 중... (500개, workers=28)
[Zephyr Z4] nodes=576, edges=5032, degree: min=10, max=20, avg=17.5
  [posiform] 5328 clauses로 유일성 확보  [posiform] 6192 clauses로 유일성 확보

  [posiform] 5040 clauses로 유일성 확보  [posiform] 5616 clauses로 유일성 확보

  [posiform] 5616 clauses로 유일성 확보
  [posiform] 6192 clauses로 유일성 확보

  [posiform] 6480 clauses로 유일성 확보  [posiform] 6624 clauses로 유일성 확보
  [posiform] 6768 clauses로 유일성 확보
  [posiform] 7776 clauses로 유일성 확보
  [posiform] 7488 clauses로 유일성 확보
  [posiform] 7344 clauses로 유일성 확보  [posiform] 7920 clauses로 유일성 확보


  [posiform] 7344 clauses로 유일성 확보  [posiform] 7632 clauses로 유일성 확보  [posiform] 7488 clauses로 유일성 확보

  [posiform] 7632 clauses로 유일성 확보
  [posiform] 8352 clauses로 유일성 확보
  [posiform] 8640 clauses로 유일성 확보
  [posiform] 8640 clauses로 유일성 확보
  [posiform] 9072 clauses로 유일성 확보

  [posiform] 9792 clauses로 유일성 확보  [posiform] 6336 clauses로 유일성 확보
  [posiform] 10080 clauses로 유일성 확보  [posiform] 10944 clauses로 유일성 확보

  

## 8. SA: GSP vs α (sweep=1000 고정)

In [19]:
# ═══ GSP vs α (sweep=1000 고정) ═══
import matplotlib.pyplot as plt
import json
import os
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

# ─── 하이퍼파라미터 ───
TOPO = 'zephyr'            # 'pegasus', 'zephyr', 'chimera'
TOPO_SIZE = 4              # pegasus=16, zephyr=4, chimera=16
COEFF = 'lin2'             # 'lin2', 'lin20'
NUM_INSTANCES = 500         # 500 or 1000

alphas_gsp = [0, 0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
fixed_sweep = 1000
num_reads_gsp = 100
NUM_WORKERS = min(os.cpu_count() - 2, 24)

# ─── 인스턴스 로드 ───
inst_list, inst_meta = load_instances(NUM_INSTANCES, topo=TOPO, topo_size=TOPO_SIZE, coeff=COEFF)
dp_mean = inst_meta['delta_p_mean']
dr_mean = inst_meta['delta_r_mean']
topo_label = f"{inst_meta['topology'].capitalize()} {inst_meta['topology'][0].upper()}{inst_meta['topo_size']}"

# ─── 결과 디렉토리 ───
base_results_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                                'hardened_posiform', 'results')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir_gsp = os.path.join(base_results_dir,
    f'gsp_vs_alpha_{inst_meta["topology"]}{inst_meta["topo_size"]}_{inst_meta["coeff"]}_inst{NUM_INSTANCES}_{timestamp}')
os.makedirs(run_dir_gsp, exist_ok=True)

print(f"═══ GSP vs α (sweep={fixed_sweep}) ═══")
print(f"  topology: {topo_label}, coeff: {inst_meta['coeff']}")
print(f"  alphas: {alphas_gsp}")
print(f"  instances: {NUM_INSTANCES}, reads: {num_reads_gsp}, workers: {NUM_WORKERS}")
print(f"  Δ_P={dp_mean:.4f}, Δ_R={dr_mean:.2f}")
print(f"  결과 디렉토리: {run_dir_gsp}")

# ─── SA worker ───
def sa_worker_gsp(args):
    import neal
    Q_dict, num_reads, num_sweeps, target_energy = args
    sampler = neal.SimulatedAnnealingSampler()
    resp = sampler.sample_qubo(Q_dict, num_reads=num_reads, num_sweeps=num_sweeps)
    best_energy = float('inf')
    eng_found = 0
    for sample, energy, _ in resp.data(['sample', 'energy', 'num_occurrences']):
        best_energy = min(best_energy, energy)
        if abs(energy - target_energy) < 1e-4:
            eng_found += 1
    return eng_found, best_energy - target_energy, num_reads

# ─── SA 실험 (병렬) ───
print(f"\nSA 실험 중...")
gsp_results = {}

t0 = time.perf_counter()
for ai, alpha in enumerate(alphas_gsp):
    tasks = []
    for inst in inst_list:
        Q_dict = dict(inst['R_sum'])
        for k, v in inst['P'].items():
            Q_dict[k] = Q_dict.get(k, 0) + alpha * v
        Q_dict = {k: v for k, v in Q_dict.items() if abs(v) > 1e-15}
        t_energy = inst['t_energy_r'] + alpha * inst['t_energy_p']
        tasks.append((Q_dict, num_reads_gsp, fixed_sweep, t_energy))

    eng_total = 0
    reads_total = 0

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = [executor.submit(sa_worker_gsp, task) for task in tasks]
        for future in as_completed(futures):
            eng_found, de_best, n_reads = future.result()
            eng_total += eng_found
            reads_total += n_reads

    gsp = eng_total / reads_total if reads_total > 0 else 0
    gsp_results[alpha] = {'gsp_eng': gsp, 'eng_found': eng_total, 'total': reads_total}

    elapsed = time.perf_counter() - t0
    eta = elapsed / (ai + 1) * (len(alphas_gsp) - ai - 1)
    print(f"  α={alpha:<6}: GSP={gsp:.3f} [{ai+1}/{len(alphas_gsp)}, {elapsed:.0f}s, ETA {eta:.0f}s]")

elapsed = time.perf_counter() - t0
print(f"\n  완료: {elapsed:.1f}s")

# ─── 시각화 ───
fig, ax = plt.subplots(figsize=(8, 5))
alpha_vals = list(alphas_gsp)
gsp_vals = [gsp_results[a]['gsp_eng'] for a in alphas_gsp]

ax.plot(range(len(alpha_vals)), gsp_vals, 'o-', color='tab:blue',
        markersize=8, linewidth=2)
ax.plot(0, gsp_vals[0], 's', color='gray', markersize=12, zorder=5,
        label='α=0 (degeneracy)')

ax.set_xticks(range(len(alpha_vals)))
ax.set_xticklabels([str(a) for a in alpha_vals], fontsize=11)
ax.set_xlabel(r'$\alpha$ (posiform scaling coefficient)', fontsize=13)
ax.set_ylabel('Energy Success Rate', fontsize=13)
ax.set_title(f'Energy GSP vs α — {topo_label}, {inst_meta["coeff"]}, sweep={fixed_sweep}', fontsize=14)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(run_dir_gsp, 'gsp_vs_alpha.png'), dpi=200)
fig.savefig(os.path.join(run_dir_gsp, 'gsp_vs_alpha.pdf'))
plt.show()

# ─── README + JSON ───
md_path = os.path.join(run_dir_gsp, 'README.md')
with open(md_path, 'w') as f:
    f.write(f"# GSP vs α 실험 결과\n\n")
    f.write(f"**날짜**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"## 실험 설정\n\n")
    f.write(f"| 항목 | 값 |\n|---|---|\n")
    f.write(f"| Topology | {topo_label} (n={inst_meta['n']}) |\n")
    f.write(f"| coeff_type | {inst_meta['coeff']} |\n")
    f.write(f"| α | {alphas_gsp} |\n")
    f.write(f"| sweep | {fixed_sweep} |\n")
    f.write(f"| num_reads | {num_reads_gsp} |\n")
    f.write(f"| num_instances | {NUM_INSTANCES} |\n")
    f.write(f"| 총 소요 시간 | {elapsed:.1f}s |\n\n")
    f.write(f"## 에너지 갭 분석\n\n")
    f.write(f"| α | α·Δ_P | Δ_R | ρ = Δ_R/(α·Δ_P) | GSP | 비고 |\n")
    f.write(f"|---|---|---|---|---|---|\n")
    for alpha in alphas_gsp:
        gsp = gsp_results[alpha]['gsp_eng']
        if alpha == 0:
            f.write(f"| 0 | 0 | {dr_mean:.2f} | — | {gsp:.1%} | 축퇴 |\n")
        else:
            adp = alpha * dp_mean
            rho = dr_mean / adp
            f.write(f"| {alpha} | {adp:.6f} | {dr_mean:.2f} | {rho:.0f} | {gsp:.1%} | |\n")
    f.write(f"\nΔ_P mean = {dp_mean:.4f}, Δ_R mean = {dr_mean:.2f}\n\n")
    f.write(f"## 파일 목록\n\n")
    f.write(f"- `gsp_vs_alpha.png/pdf` — Fig\n")
    f.write(f"- `data.json` — 전체 데이터\n")

json_path = os.path.join(run_dir_gsp, 'data.json')
with open(json_path, 'w') as f:
    json.dump({
        'params': {
            'topology': inst_meta['topology'], 'topo_size': inst_meta['topo_size'],
            'coeff': inst_meta['coeff'], 'num_instances': NUM_INSTANCES,
            'num_reads': num_reads_gsp, 'fixed_sweep': fixed_sweep,
            'alphas': alphas_gsp, 'elapsed_s': round(elapsed, 1),
        },
        'gap_stats': {'delta_p_mean': dp_mean, 'delta_r_mean': dr_mean},
        'results': {str(a): gsp_results[a] for a in alphas_gsp},
    }, f, indent=2)

print(f"  ═══ 결과: {run_dir_gsp} ═══")

FileNotFoundError: /home/yideun/qubo_dataset/hardened_posiform/hardened_posiform/instances/instances_zephyr4_lin2_1000.pkl 없음 — 먼저 생성하세요

## 9. QPU: GSP vs α (D-Wave Advantage)

### API 토큰 설정
1. https://cloud.dwavesys.com → **API Tokens** → 토큰 복사
2. 아래 셀에서 `MY_TOKEN = "DEV-여기에붙여넣기"` 로 변경

In [ ]:
# ═══ QPU: GSP vs α ═══
from dwave.system import DWaveSampler, FixedEmbeddingComposite
import matplotlib.pyplot as plt
import json
import os
from datetime import datetime

# ═══ API 토큰 ═══
MY_TOKEN = "yXwP-644aea66ae6f2920550b5c28a53d47e018fdf32e" # ← "DEV-xxxxx" 형태로 변경

# ─── 하이퍼파라미터 ───
TOPO = 'pegasus'            # 'pegasus', 'zephyr', 'chimera'
TOPO_SIZE = 16              # pegasus=16, zephyr=4, chimera=16
COEFF = 'lin20'             # 'lin2', 'lin20'
NUM_INSTANCES = 500         # 500 or 1000

alphas_qpu = [0, 0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
qpu_num_reads = 100
qpu_annealing_time = 20     # μs (D-Wave 기본값)

# ─── QPU 연결 ───
if MY_TOKEN:
    qpu = DWaveSampler(token=MY_TOKEN)
else:
    qpu = DWaveSampler()

print(f"QPU: {qpu.solver.name}")
print(f"Qubits: {qpu.solver.num_qubits}")
topo_type = qpu.properties.get('topology', {}).get('type', 'unknown')
print(f"Topology: {topo_type}")

# QPU 활성 큐빗/커플러
qpu_nodes = set(qpu.nodelist)
qpu_edges = set(map(lambda e: (min(e), max(e)), qpu.edgelist))
print(f"Active qubits: {len(qpu_nodes)}, Active couplers: {len(qpu_edges)}")

# ─── 사전 생성된 인스턴스 로드 ───
inst_list_qpu, inst_meta_qpu = load_instances(NUM_INSTANCES, topo=TOPO, topo_size=TOPO_SIZE, coeff=COEFF)
topo_label = f"{inst_meta_qpu['topology'].capitalize()} {inst_meta_qpu['topology'][0].upper()}{inst_meta_qpu['topo_size']}"

# QPU 활성 엣지로 인스턴스 호환성 검증
# 이상적 그래프로 생성된 인스턴스 중, QPU 결함 큐빗/커플러에 걸리는 것 필터링
valid_instances = []
for inst in inst_list_qpu:
    qubo_nodes = set()
    qubo_edges = set()
    for (i, j) in list(inst['R_sum'].keys()) + list(inst['P'].keys()):
        qubo_nodes.add(i)
        if i != j:
            qubo_nodes.add(j)
            qubo_edges.add((min(i, j), max(i, j)))
    if qubo_nodes.issubset(qpu_nodes) and qubo_edges.issubset(qpu_edges):
        valid_instances.append(inst)

print(f"\n  인스턴스 호환성: {len(valid_instances)}/{len(inst_list_qpu)} valid")
if len(valid_instances) < len(inst_list_qpu):
    print(f"  [Warning] {len(inst_list_qpu) - len(valid_instances)}개 인스턴스가 QPU 결함 큐빗/커플러에 걸려 제외됨")
inst_cache_qpu = valid_instances

# ─── 결과 디렉토리 ───
base_results_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                                'hardened_posiform', 'results')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir_qpu = os.path.join(base_results_dir,
    f'qpu_gsp_vs_alpha_{inst_meta_qpu["topology"]}{inst_meta_qpu["topo_size"]}_{inst_meta_qpu["coeff"]}_inst{len(inst_cache_qpu)}_{timestamp}')
os.makedirs(run_dir_qpu, exist_ok=True)

print(f"\n═══ QPU GSP vs α ═══")
print(f"  topology: {topo_label}, coeff: {inst_meta_qpu['coeff']}")
print(f"  alphas: {alphas_qpu}")
print(f"  annealing: {qpu_annealing_time}μs, reads: {qpu_num_reads}")
print(f"  instances: {len(inst_cache_qpu)}")
print(f"  결과 디렉토리: {run_dir_qpu}")

# ─── QPU 실험 ───
print(f"\nQPU 실험 중...")
qpu_results = {}

t0 = time.perf_counter()
for ai, alpha in enumerate(alphas_qpu):
    eng_total = 0
    reads_total = 0

    for inst_idx, cache in enumerate(inst_cache_qpu):
        Q_dict = dict(cache['R_sum'])
        for k, v in cache['P'].items():
            Q_dict[k] = Q_dict.get(k, 0) + alpha * v
        Q_dict = {k: v for k, v in Q_dict.items() if abs(v) > 1e-15}

        t_energy = cache['t_energy_r'] + alpha * cache['t_energy_p']

        # identity embedding
        qubo_nodes_inst = set()
        for (i, j) in Q_dict:
            qubo_nodes_inst.add(i)
            if i != j:
                qubo_nodes_inst.add(j)
        identity_emb = {v: [v] for v in qubo_nodes_inst}

        try:
            qpu_sampler = FixedEmbeddingComposite(qpu, embedding=identity_emb)
            resp = qpu_sampler.sample_qubo(Q_dict, num_reads=qpu_num_reads,
                                            annealing_time=qpu_annealing_time)
            for sample, energy, _ in resp.data(['sample', 'energy', 'num_occurrences']):
                reads_total += 1
                if abs(energy - t_energy) < 0.5:  # QPU tolerance
                    eng_total += 1
        except Exception as e:
            reads_total += qpu_num_reads
            if inst_idx == 0:
                print(f"  [Warning] α={alpha}, inst 0: {e}")

    gsp = eng_total / reads_total if reads_total > 0 else 0
    qpu_results[alpha] = {'gsp_eng': gsp, 'eng_found': eng_total, 'total': reads_total}

    elapsed = time.perf_counter() - t0
    eta = elapsed / (ai + 1) * (len(alphas_qpu) - ai - 1)
    print(f"  α={alpha:<6}: GSP={gsp:.3f} [{ai+1}/{len(alphas_qpu)}, {elapsed:.0f}s, ETA {eta:.0f}s]")

elapsed = time.perf_counter() - t0
print(f"\n  완료: {elapsed:.1f}s")

# ─── 시각화 ───
fig, ax = plt.subplots(figsize=(8, 5))

# QPU (주황)
alpha_vals = list(alphas_qpu)
qpu_gsp_vals = [qpu_results[a]['gsp_eng'] for a in alphas_qpu]
ax.plot(range(len(alpha_vals)), qpu_gsp_vals, 'o-', color='tab:orange',
        markersize=8, linewidth=2, label='QPU (D-Wave)')

# SA (파랑, 이전 실험 결과가 있으면 같이 표시)
try:
    sa_gsp_vals = [gsp_results[a]['gsp_eng'] for a in alphas_qpu]
    ax.plot(range(len(alpha_vals)), sa_gsp_vals, 's--', color='tab:blue',
            markersize=8, linewidth=2, label='SA (sweep=1000)')
except:
    print("  SA 결과 없음 — QPU만 표시")

ax.set_xticks(range(len(alpha_vals)))
ax.set_xticklabels([str(a) for a in alpha_vals], fontsize=11)
ax.set_xlabel(r'$\alpha$ (posiform scaling coefficient)', fontsize=13)
ax.set_ylabel('Energy Success Rate', fontsize=13)
ax.set_title(f'QPU vs SA — {topo_label}, {inst_meta_qpu["coeff"]}', fontsize=14)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(run_dir_qpu, 'qpu_vs_sa.png'), dpi=200)
fig.savefig(os.path.join(run_dir_qpu, 'qpu_vs_sa.pdf'))
plt.show()

# QPU 단독 그래프 (주황)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(alpha_vals)), qpu_gsp_vals, 'o-', color='tab:orange',
        markersize=8, linewidth=2)
ax.plot(0, qpu_gsp_vals[0], 's', color='gray', markersize=12, zorder=5,
        label='α=0 (degeneracy)')
ax.set_xticks(range(len(alpha_vals)))
ax.set_xticklabels([str(a) for a in alpha_vals], fontsize=11)
ax.set_xlabel(r'$\alpha$ (posiform scaling coefficient)', fontsize=13)
ax.set_ylabel('Energy Success Rate', fontsize=13)
ax.set_title(f'QPU Energy GSP vs α — {topo_label}, {inst_meta_qpu["coeff"]}', fontsize=14)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(run_dir_qpu, 'qpu_gsp_vs_alpha.png'), dpi=200)
fig.savefig(os.path.join(run_dir_qpu, 'qpu_gsp_vs_alpha.pdf'))
plt.show()

# ─── 결과 테이블 ───
print(f"\n{'α':<8} {'QPU GSP':>10}")
print('─' * 20)
for alpha in alphas_qpu:
    print(f"{alpha:<8} {qpu_results[alpha]['gsp_eng']:>9.3f}")

# ─── README ───
dp_mean_qpu = inst_meta_qpu['delta_p_mean']
dr_mean_qpu = inst_meta_qpu['delta_r_mean']

md_path_qpu = os.path.join(run_dir_qpu, 'README.md')
with open(md_path_qpu, 'w') as f:
    f.write(f"# QPU GSP vs α 실험 결과\n\n")
    f.write(f"**날짜**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"## 실험 설정\n\n")
    f.write(f"| 항목 | 값 |\n|---|---|\n")
    f.write(f"| QPU | {qpu.solver.name} |\n")
    f.write(f"| Topology | {topo_label} ({len(qpu_nodes)} active qubits) |\n")
    f.write(f"| coeff_type | {inst_meta_qpu['coeff']} |\n")
    f.write(f"| α | {alphas_qpu} |\n")
    f.write(f"| annealing_time | {qpu_annealing_time}μs |\n")
    f.write(f"| num_reads | {qpu_num_reads} |\n")
    f.write(f"| num_instances | {len(inst_cache_qpu)} (of {NUM_INSTANCES}) |\n")
    f.write(f"| energy_tol | 0.5 |\n")
    f.write(f"| 총 소요 시간 | {elapsed:.1f}s |\n\n")
    f.write(f"## 에너지 갭 분석\n\n")
    f.write(f"| α | α·Δ_P | Δ_R | ρ = Δ_R/(α·Δ_P) | QPU GSP | 비고 |\n")
    f.write(f"|---|---|---|---|---|---|\n")
    for alpha in alphas_qpu:
        gsp = qpu_results[alpha]['gsp_eng']
        if alpha == 0:
            f.write(f"| 0 | 0 | {dr_mean_qpu:.2f} | — | {gsp:.3f} | 축퇴 |\n")
        else:
            adp = alpha * dp_mean_qpu
            rho = dr_mean_qpu / adp
            f.write(f"| {alpha} | {adp:.6f} | {dr_mean_qpu:.2f} | {rho:.0f} | {gsp:.3f} | |\n")
    f.write(f"\nΔ_P mean = {dp_mean_qpu:.4f}, Δ_R mean = {dr_mean_qpu:.2f}\n\n")
    f.write(f"## 파일 목록\n\n")
    f.write(f"- `qpu_vs_sa.png/pdf` — QPU vs SA 비교\n")
    f.write(f"- `qpu_gsp_vs_alpha.png/pdf` — QPU 단독\n")
    f.write(f"- `data.json` — 전체 데이터\n")
print(f"  README: {md_path_qpu}")

# ─── JSON ───
json_qpu = os.path.join(run_dir_qpu, 'data.json')
with open(json_qpu, 'w') as f:
    json.dump({
        'params': {
            'qpu_solver': qpu.solver.name, 'topology': topo_type,
            'topo_size': inst_meta_qpu['topo_size'],
            'coeff': inst_meta_qpu['coeff'],
            'qpu_qubits': len(qpu_nodes),
            'alphas': alphas_qpu, 'annealing_time': qpu_annealing_time,
            'num_reads': qpu_num_reads,
            'num_instances': len(inst_cache_qpu),
            'num_instances_requested': NUM_INSTANCES,
            'energy_tol': 0.5, 'elapsed_s': round(elapsed, 1),
            'date': datetime.now().strftime('%Y-%m-%d %H:%M'),
        },
        'gap_stats': {'delta_p_mean': dp_mean_qpu, 'delta_r_mean': dr_mean_qpu},
        'results': {str(a): qpu_results[a] for a in alphas_qpu},
    }, f, indent=2)
print(f"  JSON: {json_qpu}")
print(f"\n  ═══ 결과 디렉토리: {run_dir_qpu} ═══")

In [ ]:
# ═══ QPU 결과에 Δ_P, Δ_R 추가 (저장된 인스턴스 사용) ═══
import json
import os

# 저장된 인스턴스에서 Δ_P, Δ_R 가져오기
inst_for_gap, meta_for_gap = load_instances(500)
dp_mean_qpu = meta_for_gap['delta_p_mean']
dr_mean_qpu = meta_for_gap['delta_r_mean']

# QPU 결과 로드
qpu_result_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')),
    'hardened_posiform', 'results', 'qpu_gsp_vs_alpha_pegasus_inst500_20260325_214219')
with open(os.path.join(qpu_result_dir, 'data.json')) as f:
    qpu_data = json.load(f)

alphas_saved = qpu_data['params']['alphas']

# README 업데이트
md_path = os.path.join(qpu_result_dir, 'README.md')
with open(md_path, 'w') as f:
    f.write(f"# QPU GSP vs α 실험 결과\n\n")
    f.write(f"**날짜**: 2026-03-25 23:35\n\n")
    f.write(f"## 실험 설정\n\n")
    f.write(f"| 항목 | 값 |\n|---|---|\n")
    f.write(f"| QPU | Advantage_system4.1 |\n")
    f.write(f"| Topology | pegasus ({qpu_data['params']['qpu_qubits']} qubits) |\n")
    f.write(f"| coeff_type | lin2 |\n")
    f.write(f"| α | {alphas_saved} |\n")
    f.write(f"| annealing_time | 20μs |\n")
    f.write(f"| num_reads | 100 |\n")
    f.write(f"| num_instances | 500 |\n")
    f.write(f"| energy_tol | 0.5 |\n\n")
    f.write(f"## 에너지 갭 분석\n\n")
    f.write(f"| α | α·Δ_P | Δ_R | ρ = Δ_R/(α·Δ_P) | QPU GSP | 비고 |\n")
    f.write(f"|---|---|---|---|---|---|\n")
    for alpha in alphas_saved:
        gsp = qpu_data['results'][str(alpha)]['gsp_eng']
        if alpha == 0:
            f.write(f"| 0 | 0 | {dr_mean_qpu:.2f} | — | {gsp:.3f} | 축퇴 |\n")
        else:
            adp = alpha * dp_mean_qpu
            rho = dr_mean_qpu / adp
            f.write(f"| {alpha} | {adp:.6f} | {dr_mean_qpu:.2f} | {rho:.0f} | {gsp:.3f} | |\n")
    f.write(f"\nΔ_P mean = {dp_mean_qpu:.4f}, Δ_R mean = {dr_mean_qpu:.2f}\n\n")
    f.write(f"## 파일 목록\n\n")
    f.write(f"- `qpu_vs_sa.png/pdf` — QPU vs SA 비교\n")
    f.write(f"- `qpu_gsp_vs_alpha.png/pdf` — QPU 단독\n")
    f.write(f"- `qpu_gsp_vs_alpha_zoomed.png/pdf` — QPU 단독 (y축 확대)\n")
    f.write(f"- `data.json` — 전체 데이터\n")

print(f"  README 업데이트 완료: {md_path}")
print(f"  Δ_P={dp_mean_qpu:.4f}, Δ_R={dr_mean_qpu:.2f}")